In [ ]:
import torch
import os
from torch.utils.data import DataLoader
from torchvision.models import vgg19, VGG19_Weights
from tqdm import tqdm
from PIL import Image
from torchvision.utils import save_image
from IPython.display import Image as IPythonImage, display
from torchvision import transforms
import matplotlib.pyplot as plt

# Import YOUR custom files!
from networks import Generator, Discriminator
from dataset import SRDataset
from losses import GLoss, DLoss

In [ ]:
class VGGFeatureExtractor(torch.nn.Module):

    def __init__(self,device):
        super().__init__()

        vgg = vgg19(weights=VGG19_Weights.IMAGENET1K_V1).to(device)
        vgg = vgg.features
        feature_extractor = torch.nn.Sequential( *list(vgg.children())[:35] )

        for p in feature_extractor.parameters():
            p.requires_grad = False

        feature_extractor.eval()

        self.feature_extractor = feature_extractor
        self.mean = torch.tensor([0.485, 0.456, 0.406],device=device).view(1,3,1,1)
        self.std = torch.tensor([0.229, 0.224, 0.225],device=device).view(1,3,1,1)

    def forward(self,x):
        x = (x-self.mean)/self.std
        return self.feature_extractor(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = SRDataset("div2k\DIV2K_train_HR",patch_size=256,scale=4)
dataloader = DataLoader(dataset,4,True,num_workers=0,pin_memory=True)

G = Generator(64,23,32,5,4).to(device)
D = Discriminator().to(device)
feature_extractor = VGGFeatureExtractor(device)

print("VGG19 setup completed")
print("Generator , Discriminator Loaded")
print(f"DIV2K data loaded, batch size: {4}")
print(device)

In [ ]:
L1 = torch.nn.L1Loss()
BCEloss = torch.nn.BCEWithLogitsLoss()

optimizer_G = torch.optim.Adam(
    G.parameters(),
    lr=2e-4,
    betas=(0.9, 0.999)
)

optimizer_D = torch.optim.Adam(
    D.parameters(),
    lr=2e-4,
    betas=(0.9, 0.999)
)

In [ ]:
import torch
import os
from torchvision.utils import save_image
from tqdm import tqdm # Used for a nice progress bar

os.makedirs("stage1_progress", exist_ok=True)

L1 = torch.nn.L1Loss()
BCEloss = torch.nn.BCEWithLogitsLoss()

optimizer_G_pretrain = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.9, 0.999))
optimizer_G = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.9, 0.999))
optimizer_D = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.9, 0.999))


epochs = 20
for epoch in range(epochs):
    loop = tqdm(dataloader, desc=f"Epoch [{epoch+1}/{epochs}]")
    
    epoch_g_loss = 0.0

    for lr, hr in loop:
        lr = lr.to(device)
        hr = hr.to(device)

        optimizer_G_pretrain.zero_grad()

        sr = G(lr)

        loss = L1(sr,hr)

        epoch_g_loss += loss.item()

        loss.backward()
        optimizer_G_pretrain.step()
        loop.set_postfix(L1_loss=loss.item())
    
    avg_g_loss = epoch_g_loss/len(dataloader)

    print(f"End of epoch : {epoch+1} | average Geneterator loss : {avg_g_loss}")
    
    with torch.no_grad():
        lr_upscaled = torch.nn.functional.interpolate(lr, size=hr.shape[2:], mode='bicubic')
        comparison_grid = torch.cat([lr_upscaled[:2], sr[:2], hr[:2]], dim=0)
        
        img_path = f"stage1_progress/epoch_{epoch+1}.png"
        save_image(comparison_grid, img_path, nrow=2, normalize=True)
        
        print(f"Visualizing Stage 1 - Epoch {epoch+1}:")
        display(IPythonImage(filename=img_path))

torch.save(G.state_dict(), "generator_psnr.pth")
print("Stage 1 Complete! The checkerboards should be gone.")